# Week 5: RAG and Information Retrieval with SPLADE

In this practical, we will build a complete RAG (Retrieval-Augmented Generation)
system using the LoTTE dataset from ir-datasets.

We use PyTerrier's integration with ir-datasets for easy access:
https://ir-datasets.com/pyterrier.html

**Note**: This practical can optionally use synthetic training data generated
in Practical 04 (`training_data_for_splade.json`). Run Practical 04 first
to generate more training data for better SPLADE performance.

Outline:
1. **IR Fundamentals**: Indexing and BM25 with PyTerrier
1b. **Hard Negative Mining**: Creating effective training data
2. **Introduction to SPLADE**: Learned sparse representations
3. **SPLADE Implementation**: Architecture from scratch
4. **SPLADE Training**: Contrastive learning with hard negatives
5. **Distillation**: Cross-encoder to bi-encoder
6. **RAG Pipeline**: Retrieval + Generation
7. **Evaluation**: MRR, Recall@k

## Setup

## Load the Dataset

We load the LoTTE dataset directly using PyTerrier's ir-datasets integration.
This gives us access to documents, queries, and relevance judgments.

See: https://ir-datasets.com/pyterrier.html

In [ ]:
import pandas as pd
import pyterrier as pt
import torch
import torch.nn as nn
import torch.nn.functional as F
import ir_measures
import getpass
import os
import tarfile
import urllib.request
from pathlib import Path
from typing import Dict, List, Tuple
from ir_measures import RR, Recall


In [ ]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

In [ ]:
"""Cache utilities for downloading and caching files."""

In [ ]:
def get_from_cache(url: str, name: str) -> Path:
    """Get a file from cache, downloading it if necessary.

    Args:
        url: URL to download the file from if not cached.
        name: Name of the cached file.

    Returns:
        Path to the cached file.
    """
    cache_dir_env = os.environ.get("MASTER_MIND_CACHE")

    if cache_dir_env:
        # Check if file exists in the cache directory
        cache_path = Path(cache_dir_env) / name
        if cache_path.exists():
            print(f"Found {cache_path}")
            return cache_path

        # File doesn't exist, create temp directory and download
        username = getpass.getuser()
        temp_dir = Path(f"/tmp/{username}/master-mind")
        print(f"Did not find {name} in ${cache_dir_env}, downloading to {temp_dir}...")
        temp_dir.mkdir(parents=True, exist_ok=True)
        cache_path = temp_dir / name
    else:
        # No cache env variable, use outputs/cache
        cache_dir = Path("outputs/cache")
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = cache_dir / name

    # Download if file doesn't exist
    if not cache_path.exists():
        print(f"Loading {url} to cache {cache_path}...")
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if url.endswith(".tar.gz"):
            # Download to temp file and extract
            archive_path = cache_path.with_suffix(".tar.gz")
            urllib.request.urlretrieve(url, archive_path)
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(path=cache_path)
            archive_path.unlink()
        else:
            urllib.request.urlretrieve(url, cache_path)

    return cache_path

In [ ]:
# Load the LoTTE technology dataset via PyTerrier's ir-datasets integration
dataset = pt.get_dataset("irds:lotte/technology/dev/search")
print("Dataset loaded: lotte/technology/dev/search")

In [ ]:
# Configuration
num_docs = 5000  # Limit documents for practical
num_queries = 100  # Limit queries

In [ ]:
# Get queries and qrels first to know which documents are relevant
queries_df = dataset.get_topics().head(num_queries)
qrels_df = dataset.get_qrels()

# Filter qrels for our queries first
query_ids = set(queries_df["qid"].tolist())
qrels_df = qrels_df[qrels_df["qid"].isin(query_ids)].copy()

# Get the set of relevant document IDs we need
relevant_doc_ids = set(qrels_df["docno"].tolist())

print("Queries and Qrels loaded:")
print(f"  - Queries: {len(queries_df)}")
print(f"  - Qrels: {len(qrels_df)}")

## Part 1: Information Retrieval Fundamentals

### Indexing with PyTerrier

`IterDictIndexer` accepts any iterable of `{"docno": ..., "text": ...}` dicts.
We configure `meta` to store document text in the index for later retrieval.
This allows scaling to larger collections without keeping all text in memory.

In [ ]:
# Download the PyTerrier index of Lotte (technology subset)
index_path = get_from_cache(
    "https://master-mind.isir.upmc.fr/llm/data/index_lotte.tar.gz", "llm/index_lotte"
)

# Initialize PyTerrier with the index
index = pt.terrier.TerrierIndex(str(index_path))

In [ ]:
def get_text_from_index(
    index: pt.terrier.TerrierIndex, doc_ids: List[str]
) -> Dict[str, str]:
    """
    Retrieve document text from the index metadata.

    Args:
        index: PyTerrier index reference
        doc_ids: List of document IDs

    Returns:
        Dict mapping doc_id -> text
    """
    meta_index = index.meta_index()

    result = {}
    for doc_id in doc_ids:
        try:
            # Get internal docid from docno
            docid = meta_index.getDocument("docno", doc_id)
            if docid >= 0:
                text = meta_index.getItem("text", docid)
                result[doc_id] = text
        except Exception:
            pass  # Document not found

    return result


# Helper for single document lookup
def get_doc_text(index, doc_id: str) -> str:
    """Get text for a single document from the index."""
    texts = get_text_from_index(index, [doc_id])
    return texts.get(doc_id, "")


def doc_exists_in_index(index, doc_id: str) -> bool:
    """Check if a document exists in the index (without fetching text)."""
    meta_index = index.meta_index()
    try:
        return meta_index.getDocument("docno", doc_id) >= 0
    except Exception:
        return False

### BM25 Baseline Retriever

In [ ]:
# Create a BM25 retriever
bm25 = index.retriever("BM25")

In [ ]:
# Test search
test_query = "How do I install Python packages?"
results = bm25.search(test_query)
print(f"Top 3 results for: '{test_query}'")
for i, row in results.head(3).iterrows():
    doc_text = get_doc_text(index, row["docno"])
    print(f"\n[{i + 1}] Score: {row['score']:.2f}")
    print(f"    Doc: {doc_text[:150]}...")

### Evaluation with MRR and Recall@k

We use `ir-measures` to compute standard IR metrics.

In [ ]:
# Convert PyTerrier dataframes to ir-measures format
def to_ir_measures_qrels(qrels_df):
    """Convert PyTerrier qrels to ir-measures format."""
    return qrels_df.rename(
        columns={"qid": "query_id", "docno": "doc_id", "label": "relevance"}
    )


def to_ir_measures_run(run_df):
    """Convert PyTerrier run to ir-measures format."""
    return run_df.rename(columns={"qid": "query_id", "docno": "doc_id"})


qrels_ir = to_ir_measures_qrels(qrels_df)

In [ ]:
# Evaluate BM25
results = bm25.transform(queries_df[["qid", "query"]])

# Compute metrics using ir-measures
metrics = [RR, Recall @ 1, Recall @ 5, Recall @ 10]
eval_results = ir_measures.calc_aggregate(
    metrics, qrels_ir, to_ir_measures_run(results)
)

print("BM25 Baseline:")
for metric, value in eval_results.items():
    print(f"  {metric}: {value:.4f}")

## Part 1b: Hard Negative Mining

To train a good retrieval model, we need more than just positive examples.
**Hard negatives** are documents that seem relevant but are not the ground truth.

Why hard negatives matter:
- **Random negatives** are too easy to distinguish → model doesn't learn much
- **Hard negatives** force the model to learn subtle differences
- BM25 provides good hard negatives: similar in lexical terms, but different in meaning

### Hard Negative Mining

In [ ]:
def find_hard_negatives_bm25(
    query: str,
    positive_doc_ids: set,
    retriever,
    num_negatives: int = 5,
) -> List[str]:
    """
    Find hard negatives using BM25.

    Hard negatives are documents that are lexically similar to the query
    but are NOT the ground truth positive document.

    Args:
        query: The query text
        positive_doc_ids: Set of positive document IDs (to exclude)
        retriever: BM25 retriever
        num_negatives: Number of hard negatives to return

    Returns:
        List of document IDs for hard negatives
    """
    # Implement hard negative mining with BM25

    # 1. Retrieve top documents with BM25
    # 2. Filter out the positive documents
    # 3. Return the top-k remaining as hard negatives
    results = retriever.search(query)
    hard_negatives = []
    for _, row in results.iterrows():
    assert False, 'Not implemented yet'

In [ ]:
# Test hard negative mining
test_qid = qrels_df["qid"].iloc[0]
test_query_row = queries_df[queries_df["qid"] == test_qid].iloc[0]
test_query_text = test_query_row["query"]
test_positives = set(qrels_df[qrels_df["qid"] == test_qid]["docno"].tolist())

hard_negs = find_hard_negatives_bm25(
    test_query_text, test_positives, bm25, num_negatives=3
)

print(f"Query: {test_query_text}")
print(f"\nPositive documents ({len(test_positives)}):")
pos_texts = get_text_from_index(index, list(test_positives)[:2])
for pos_id, pos_text in pos_texts.items():
    print(f"  {pos_text[:100]}...")
print(f"\nHard negatives ({len(hard_negs)}):")
neg_texts = get_text_from_index(index, hard_negs)
for neg_id in hard_negs:
    print(f"  {neg_texts.get(neg_id, '')[:100]}...")

In [ ]:
def create_training_triplets(
    queries_df: pd.DataFrame,
    qrels_df: pd.DataFrame,
    index,
    retriever,
    num_hard_negatives: int = 3,
    max_queries: int = None,
) -> List[Dict]:
    """
    Create training triplets with hard negatives.

    Each triplet contains:
    - query: The question
    - positive: A relevant document
    - negatives: List of hard negative documents

    Args:
        queries_df: DataFrame with queries
        qrels_df: DataFrame with relevance judgments
        index: PyTerrier index reference (for text retrieval)
        retriever: BM25 retriever for hard negative mining
        num_hard_negatives: Number of hard negatives per query
        max_queries: Maximum number of queries to process

    Returns:
        List of training triplets
    """
    triplets = []

    queries_to_process = queries_df.head(max_queries) if max_queries else queries_df

    for _, query_row in tqdm(
        queries_to_process.iterrows(),
        desc="Creating training triplets",
        total=len(queries_to_process),
    ):
        qid = query_row["qid"]
        query_text = query_row["query"]

        # Get positive documents for this query
        positive_doc_ids = set(qrels_df[qrels_df["qid"] == qid]["docno"].tolist())

        if not positive_doc_ids:
            continue

        # Get the first positive document
        positive_id = list(positive_doc_ids)[0]
        positive_text = get_doc_text(index, positive_id)

        if not positive_text:
            continue

        # Find hard negatives
        hard_neg_ids = find_hard_negatives_bm25(
            query_text, positive_doc_ids, retriever, num_hard_negatives
        )
        hard_neg_texts = get_text_from_index(index, hard_neg_ids)
        hard_neg_list = [
            hard_neg_texts[nid] for nid in hard_neg_ids if nid in hard_neg_texts
        ]

        if hard_neg_list:
            triplets.append(
                {
                    "query": query_text,
                    "positive": positive_text,
                    "negatives": hard_neg_list,
                }
            )

    return triplets

### Loading synthetic training data from Practical 04

In the previous practical, we generated synthetic queries using an LLM.
We can load and use this additional training data.

In [ ]:
import json

synthetic_training_path = Path("./outputs/practical-04/training_data_for_splade.json")
synthetic_pairs = []

if synthetic_training_path.exists():
    with open(synthetic_training_path, "r", encoding="utf-8") as f:
        synthetic_data = json.load(f)

    # Convert to query-positive pairs (filter to docs in our index)
    for item in synthetic_data:
        # Check if doc exists in index
        if doc_exists_in_index(index, item["doc_id"]):
            synthetic_pairs.append(
                {
                    "query": item["query"],
                    "doc_id": item["doc_id"],
                    "source": item.get("source", "synthetic"),
                }
            )

    print(f"Loaded {len(synthetic_pairs)} synthetic training pairs from Practical 04")
else:
    print(f"No synthetic training data found at {synthetic_training_path}")
    print(
        "Run Practical 04 first to generate synthetic queries, or continue with qrels only."
    )

In [ ]:
# Create training data with hard negatives
num_train_queries = 50


# Create triplets from qrels (original data)
training_triplets = create_training_triplets(
    queries_df,
    qrels_df,
    index,
    bm25,
    num_hard_negatives=3,
    max_queries=num_train_queries,
)

print(f"\nTraining triplets from qrels: {len(training_triplets)}")

In [ ]:
# Add triplets from synthetic pairs (if available)
if synthetic_pairs:
    print(f"Adding triplets from {len(synthetic_pairs)} synthetic pairs...")

    for pair in tqdm(
        synthetic_pairs[:num_train_queries], desc="Processing synthetic pairs"
    ):
        query_text = pair["query"]
        positive_id = pair["doc_id"]
        positive_text = get_doc_text(index, positive_id)

        if not positive_text:
            continue

        # Find hard negatives for synthetic queries too
        hard_neg_ids = find_hard_negatives_bm25(
            query_text, {positive_id}, bm25, num_negatives=3
        )
        hard_neg_texts = get_text_from_index(index, hard_neg_ids)
        hard_neg_list = [
            hard_neg_texts[nid] for nid in hard_neg_ids if nid in hard_neg_texts
        ]

        if hard_neg_list:
            training_triplets.append(
                {
                    "query": query_text,
                    "positive": positive_text,
                    "negatives": hard_neg_list,
                }
            )

    print(f"Total training triplets (qrels + synthetic): {len(training_triplets)}")
if training_triplets:
    print("Example triplet:")
    print(f"  Query: {training_triplets[0]['query'][:60]}...")
    print(f"  Positive: {training_triplets[0]['positive'][:60]}...")
    print(f"  Num negatives: {len(training_triplets[0]['negatives'])}")

## Part 2: Introduction to SPLADE

SPLADE (SParse Lexical AnD Expansion) learns sparse representations
using a masked language model (MLM).

### Key idea:
- Use MLM logits to "expand" terms
- Example: "car" can activate "automobile", "vehicle", "transport"
- Sparsify with ReLU + log for few active terms

### Formula:
For each token position, we compute:
```
w_j = max_i log(1 + ReLU(MLM_logits[i, j]))
```
Where j is the vocabulary index.

### SPLADE Introduction

In [ ]:
# Load an MLM model (DistilBERT for efficiency)

# You can also use
# splade_baseline_model="naver/splade_v2_distil"

splade_baseline_model = "distilbert-base-uncased"
mlm_tokenizer = AutoTokenizer.from_pretrained(splade_baseline_model)
mlm_model = AutoModelForMaskedLM.from_pretrained(splade_baseline_model)
mlm_model = mlm_model.to(device)
mlm_model.eval()

print(f"MLM model loaded: {splade_baseline_model} on {device}")
print(f"Vocabulary size: {mlm_tokenizer.vocab_size}")

In [ ]:
def compute_splade_representation(
    text: str,
    model: AutoModelForMaskedLM,
    tokenizer: AutoTokenizer,
) -> torch.Tensor:
    """
    Compute the SPLADE representation of a text.

    Args:
        text: Text to encode
        model: MLM model
        tokenizer: Tokenizer

    Returns:
        Sparse vector of size vocab_size
    """
    # Implement SPLADE representation

    # 2. Get MLM logits
    # 3. Apply ReLU then log(1 + x)
    # 4. Max-pooling over positions
    inputs = tokenizer(text, return_tensors="pt", ...).to(device)
    with torch.no_grad():
    Implement compute_splade_representation

    assert False, 'Not implemented yet'


In [ ]:
# Test SPLADE representation
test_text = "How do I install Python packages using pip?"
sparse_rep = compute_splade_representation(test_text, mlm_model, mlm_tokenizer)

print(f"Dimension: {sparse_rep.shape}")
print(f"Non-zero terms: {(sparse_rep > 0).sum().item()}")
print(f"Sparsity: {100 * (sparse_rep == 0).sum().item() / len(sparse_rep):.1f}%")

# Display most activated terms
top_indices = torch.topk(sparse_rep, k=15).indices
top_tokens = mlm_tokenizer.convert_ids_to_tokens(top_indices.tolist())
top_values = sparse_rep[top_indices].tolist()

print("\nMost activated terms:")
for token, value in zip(top_tokens, top_values):
    print(f"  {token}: {value:.3f}")

## Part 3: SPLADE Architecture from Scratch

We will implement a complete SPLADE encoder.

### SPLADE Architecture

In [ ]:
class SPLADEEncoder(nn.Module):
    """
    SPLADE encoder based on an MLM model.

    Architecture:
    - Backbone: Pre-trained MLM model (DistilBERT, etc.)
    - Sparsification: ReLU + log(1 + x)
    - Pooling: Max over positions
    """

    def __init__(
        self,
        model_name: str = "distilbert-base-uncased",
        sparsity_weight: float = 0.0001,
    ):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name)
        self.sparsity_weight = sparsity_weight
        self.vocab_size = self.tokenizer.vocab_size

    def encode(
        self,
        texts: List[str],
        max_length: int = 256,
    ) -> torch.Tensor:
        """
        Encode a list of texts into SPLADE representations.

        Args:
            texts: List of texts
            max_length: Maximum length

        Returns:
            Tensor of shape (batch_size, vocab_size)
        """
        # Implement batch encoding

        # 2. Forward pass through MLM model
        # 3. Sparsification: ReLU + log(1 + x)
        # 4. Max-pooling over positions (attention to mask!)
        inputs = self.tokenizer(texts, return_tensors="pt", ...)
        outputs = self.model(**inputs)
        ...
        Implement the encode method

        assert False, 'Not implemented yet'


    def compute_flops_loss(self, sparse_rep: torch.Tensor) -> torch.Tensor:
        """
        Compute FLOPS penalty to encourage sparsity.

        FLOPS = sum_j (sum_i w_ij)^2

        This penalty encourages sparse representations.
        """
        # Sum over batch then square
        flops = (sparse_rep.sum(dim=0) ** 2).sum()
        return self.sparsity_weight * flops

    def forward(
        self,
        query_texts: List[str],
        doc_texts: List[str],
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass for training.

        Returns:
            query_rep: Query representations
            doc_rep: Document representations
            flops_loss: Sparsity penalty
        """
        query_rep = self.encode(query_texts)
        doc_rep = self.encode(doc_texts)

        flops_loss = self.compute_flops_loss(query_rep) + self.compute_flops_loss(
            doc_rep
        )

        return query_rep, doc_rep, flops_loss

In [ ]:
# Test the encoder
splade_encoder = SPLADEEncoder(splade_baseline_model)
splade_encoder = splade_encoder.to(device)
print(f"SPLADE encoder loaded on {device}")

test_queries = ["How to install Python packages?"]
test_docs = ["Use pip install to install Python packages from PyPI."]

with torch.no_grad():
    q_rep, d_rep, flops = splade_encoder(test_queries, test_docs)

print(f"Query rep shape: {q_rep.shape}")
print(f"Doc rep shape: {d_rep.shape}")
print(f"FLOPS loss: {flops.item():.4f}")

# Similarity score (dot product)
score = (q_rep * d_rep).sum().item()
print(f"Query-doc score: {score:.4f}")

In [ ]:
# Visualize activated terms
def visualize_splade_terms(
    rep: torch.Tensor,
    tokenizer: AutoTokenizer,
    top_k: int = 20,
):
    """Display most activated terms."""
    top_indices = torch.topk(rep, k=top_k).indices
    top_tokens = tokenizer.convert_ids_to_tokens(top_indices.tolist())
    top_values = rep[top_indices].tolist()

    print("Activated terms:")
    for token, value in zip(top_tokens, top_values):
        bar = "█" * int(value * 5)
        print(f"  {token:20s} {value:6.3f} {bar}")


print("=== Query ===")
visualize_splade_terms(q_rep[0], splade_encoder.tokenizer)
print("\n=== Document ===")
visualize_splade_terms(d_rep[0], splade_encoder.tokenizer)

## Part 4: Training SPLADE

We use a contrastive loss (InfoNCE) to train SPLADE.

For each query q:
- Positive: the associated document d+
- Negatives: other documents from the batch (in-batch negatives)

### Training SPLADE

In [ ]:
def contrastive_loss(
    query_rep: torch.Tensor,
    doc_rep: torch.Tensor,
    temperature: float = 0.05,
) -> torch.Tensor:
    """
    Compute InfoNCE contrastive loss.

    Args:
        query_rep: (batch_size, vocab_size)
        doc_rep: (batch_size, vocab_size)
        temperature: Temperature for softmax

    Returns:
        Average loss
    """
    # Implement contrastive loss

    # 1. Compute query-doc scores (dot product)
    # 2. Positives are on the diagonal
    # 3. InfoNCE: -log(exp(score_pos) / sum(exp(scores)))
    # Normalize for stability (optional but recommended)
    assert False, 'Not implemented yet'

In [ ]:
# Test the loss
batch_q = splade_encoder.encode(["Question 1", "Question 2"])
batch_d = splade_encoder.encode(["Answer 1", "Answer 2"])

loss = contrastive_loss(batch_q, batch_d)
print(f"Contrastive loss: {loss.item():.4f}")

In [ ]:
def train_splade(
    encoder: SPLADEEncoder,
    train_data: List[Dict],
    num_epochs: int = 3,
    batch_size: int = 8,
    learning_rate: float = 2e-5,
    temperature: float = 0.05,
    use_hard_negatives: bool = False,
):
    """
    Train the SPLADE encoder.

    Args:
        encoder: SPLADE model
        train_data: List of triplets {query, positive, negatives}
        num_epochs: Number of epochs
        batch_size: Batch size
        learning_rate: Learning rate
        temperature: Temperature for InfoNCE
        use_hard_negatives: If True, expects triplets with 'negatives' field
    """
    from torch.optim import AdamW

    optimizer = AdamW(encoder.parameters(), lr=learning_rate)
    encoder.train()

    for epoch in range(num_epochs):
        total_loss = 0
        total_contrastive = 0
        total_flops = 0
        num_batches = 0

        # Shuffle data
        import random

        shuffled = train_data.copy()
        random.shuffle(shuffled)

        # Create batches
        for i in tqdm(range(0, len(shuffled), batch_size), desc=f"Epoch {epoch + 1}"):
            batch = shuffled[i : i + batch_size]
            if len(batch) < 2:  # Need at least 2 for contrastive
                continue

            if use_hard_negatives:
                # Use triplets with hard negatives
                queries = [ex["query"] for ex in batch]
                positives = [ex["positive"][:500] for ex in batch]
                # Collect hard negatives from all examples in batch
                all_negatives = []
                for ex in batch:
                    all_negatives.extend([n[:500] for n in ex.get("negatives", [])[:2]])

                # Encode queries
                query_rep = encoder.encode(queries)
                # Encode positives + negatives together
                all_docs = positives + all_negatives
                doc_reps = encoder.encode(all_docs)
                positive_rep = doc_reps[: len(positives)]
                negative_rep = doc_reps[len(positives) :]

                # Contrastive loss with in-batch + hard negatives
                cont_loss = contrastive_loss(query_rep, positive_rep, temperature)

                # Add hard negative loss if we have negatives
                if len(negative_rep) > 0:
                    # Compute scores with negatives and ensure they're lower
                    neg_scores = torch.matmul(
                        F.normalize(query_rep, p=2, dim=-1),
                        F.normalize(negative_rep, p=2, dim=-1).T,
                    )
                    # Margin loss: positive scores should be higher than negative
                    hard_neg_loss = F.relu(neg_scores.mean() + 0.2).mean()
                    cont_loss = cont_loss + 0.3 * hard_neg_loss

                flops_loss = encoder.compute_flops_loss(
                    query_rep
                ) + encoder.compute_flops_loss(doc_reps)
            else:
                # Standard in-batch negatives only
                queries = [ex["query"] for ex in batch]
                docs = [ex["positive"][:500] for ex in batch]

                # Forward
                query_rep, doc_rep, flops_loss = encoder(queries, docs)

                # Contrastive loss
                cont_loss = contrastive_loss(query_rep, doc_rep, temperature)

            # Total loss
            loss = cont_loss + flops_loss

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_contrastive += cont_loss.item()
            total_flops += flops_loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        avg_cont = total_contrastive / num_batches
        avg_flops = total_flops / num_batches

        print(
            f"Epoch {epoch + 1}: Loss={avg_loss:.4f} (Contrastive={avg_cont:.4f}, FLOPS={avg_flops:.4f})"
        )

    encoder.eval()
    print("Training complete!")

In [ ]:
# Train SPLADE with hard negatives (reduced version for the practical)
num_epochs = 2


# Use training triplets created in Part 1b (with hard negatives)
if training_triplets:
    print(f"Training on {len(training_triplets)} triplets with hard negatives...")
    train_splade(
        splade_encoder,
        training_triplets,
        num_epochs=num_epochs,
        batch_size=8,
        use_hard_negatives=True,
    )
else:
    print("No training triplets available. Skipping SPLADE training.")

## Part 4b: Distillation Cross-Encoder → SPLADE

Cross-encoders are more accurate but don't allow pre-indexing.
We can distill their knowledge to a bi-encoder (SPLADE).

### Principle:
1. Train/load a cross-encoder teacher
2. The teacher scores (query, doc) pairs
3. The student (SPLADE) learns to reproduce these scores

### Distillation: Cross-Encoder to SPLADE

In [ ]:
class CrossEncoderTeacher(nn.Module):
    """
    Cross-encoder for scoring query-document pairs.

    Unlike the bi-encoder, the cross-encoder sees query and doc together,
    allowing richer interactions but preventing pre-computation.
    """

    def __init__(self, model_name: str = "distilbert-base-uncased"):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

    def forward(
        self,
        queries: List[str],
        documents: List[str],
    ) -> torch.Tensor:
        """
        Score query-document pairs.

        Args:
            queries: List of queries
            documents: List of documents (same length as queries)

        Returns:
            Relevance scores (batch_size,)
        """
        # Implement cross-encoder forward

        # 1. Concatenate query and document with [SEP]
        # 2. Tokenize
        # 3. Get [CLS] embedding
        # 4. Classify to get score
        pairs = [q + " [SEP] " + d for q, d in zip(queries, documents)]
        inputs = self.tokenizer(pairs, ...)
        outputs = self.model(**inputs)
        cls_output = outputs.last_hidden_state[:, 0, :]
        scores = self.classifier(cls_output)
        assert False, 'Not implemented yet'

In [ ]:
# Initialize cross-encoder
# Using pre-trained cross-encoder from sentence-transformers
# Lightweight options:
# - cross-encoder/ms-marco-MiniLM-L6-v2 (22.7M params, recommended)
# - cross-encoder/ms-marco-MiniLM-L12-v2 (33.4M params, more powerful)
# - cross-encoder/ms-marco-TinyBERT-L2-v2 (4.39M params, ultra-light)
cross_encoder = CrossEncoderTeacher("cross-encoder/ms-marco-MiniLM-L6-v2")
cross_encoder = cross_encoder.to(device)
print(f"Cross-encoder loaded on {device}")

# Test
test_scores = cross_encoder(
    ["How to install packages?"],
    ["Use pip install to install Python packages."],
)
print(f"Cross-encoder score: {test_scores.item():.4f}")

In [ ]:
def distillation_loss(
    student_scores: torch.Tensor,
    teacher_scores: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    Compute distillation loss (MSE or KL divergence).

    Args:
        student_scores: Student (SPLADE) scores
        teacher_scores: Teacher (cross-encoder) scores
        temperature: Temperature for softening distributions (KL only)

    Returns:
        Distillation loss
    """
    # Simple MSE between scores
    return F.mse_loss(student_scores, teacher_scores.detach())

## Part 5: Complete RAG Pipeline

Now let's combine the SPLADE retriever with a generator (SmolLM2).

### RAG Pipeline

In [ ]:
# Load generation model
#
# Options (uncomment ONE gen_model_name):
#
# 1. SmolLM2-1.7B float16 (~3.4GB) - default, works on all platforms
gen_model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
#
# 2. Pre-quantized models (GPTQ/AWQ) - Linux only
# gen_model_name = "Qwen/Qwen2.5-3B-Instruct-AWQ"  # 3B AWQ, ~2GB
# gen_model_name = "Qwen/Qwen2.5-7B-Instruct-AWQ"  # 7B AWQ, ~4GB

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)

# Detect if model is pre-quantized (AWQ/GPTQ) by name
is_quantized = "AWQ" in gen_model_name or "GPTQ" in gen_model_name

if is_quantized:
    gen_model = AutoModelForCausalLM.from_pretrained(
        gen_model_name,
        device_map="auto",
    )
    print(f"Generation model loaded: {gen_model_name} (pre-quantized)")
else:
    gen_model = AutoModelForCausalLM.from_pretrained(
        gen_model_name,
        dtype=torch.float16,
    )
    gen_model = gen_model.to(device)
    print(f"Generation model loaded: {gen_model_name} on {device}")

gen_model.eval()

In [ ]:
class RAGPipeline:
    """
    Complete RAG pipeline: Retrieval + Generation.

    Uses PyTerrier index for document storage (scalable to large collections).
    """

    def __init__(
        self,
        retriever: SPLADEEncoder,
        generator,
        generator_tokenizer,
        pt_index,
        doc_ids: List[str],
        top_k: int = 3,
    ):
        self.retriever = retriever
        self.generator = generator
        self.gen_tokenizer = generator_tokenizer
        self.pt_index = pt_index
        self.doc_ids = doc_ids
        self.top_k = top_k

        # Pre-compute document representations
        self._index_documents()

    def _get_doc_text(self, doc_id: str) -> str:
        """Get document text from PyTerrier index."""
        return get_doc_text(self.pt_index, doc_id)

    def _get_texts(self, doc_ids: List[str]) -> Dict[str, str]:
        """Get multiple document texts from PyTerrier index."""
        return get_text_from_index(self.pt_index, doc_ids)

    def _index_documents(self, batch_size: int = 16):
        """Index all documents with SPLADE."""
        print("Indexing documents with SPLADE...")

        # Get texts from PyTerrier index in batches
        all_reps = []

        with torch.no_grad():
            for i in tqdm(range(0, len(self.doc_ids), batch_size)):
                batch_ids = self.doc_ids[i : i + batch_size]
                batch_texts = self._get_texts(batch_ids)
                # Truncate texts and maintain order
                texts = [batch_texts.get(doc_id, "")[:500] for doc_id in batch_ids]
                reps = self.retriever.encode(texts)
                all_reps.append(reps.cpu())

        self.doc_reps = torch.cat(all_reps, dim=0)
        print(f"Documents indexed: {len(self.doc_ids)}")

    def retrieve(self, query: str, top_k: int = None) -> List[Tuple[str, float]]:
        """
        Retrieve most relevant documents.

        Args:
            query: User question
            top_k: Number of documents to retrieve

        Returns:
            List of (doc_id, score)
        """
        if top_k is None:
            top_k = self.top_k

        # Encode query
        with torch.no_grad():
            query_rep = self.retriever.encode([query]).cpu()

        # Compute scores
        scores = torch.matmul(query_rep, self.doc_reps.T).squeeze(0)

        # Top-k
        top_indices = torch.topk(scores, k=min(top_k, len(scores))).indices

        results = []
        for idx in top_indices:
            doc_id = self.doc_ids[idx]
            score = scores[idx].item()
            results.append((doc_id, score))

        return results

    def generate(
        self,
        query: str,
        context: str,
        max_new_tokens: int = 200,
    ) -> str:
        """Generate a response based on context."""
        system_msg = "You are a helpful assistant that answers technical questions using the provided context."
        user_msg = f"Context:\n{context}\n\nQuestion: {query}"

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        prompt = self.gen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.gen_tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = self.generator.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.gen_tokenizer.eos_token_id,
            )

        response = self.gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the response
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()

        return response

    def __call__(self, query: str) -> Dict:
        """
        Complete pipeline: retrieval + generation.

        Args:
            query: User question

        Returns:
            Dict with retrieved_docs and generated_answer
        """
        # Implement RAG pipeline

        # 1. Retrieve top-k documents
        # 2. Build context from documents
        # 3. Generate response
        retrieved = self.retrieve(query)
        doc_texts = self._get_texts([doc_id for doc_id, _ in retrieved])
        context = "\n\n".join([doc_texts[doc_id][:300] for doc_id, _ in retrieved])
        answer = self.generate(query, context)
        assert False, 'Not implemented yet'


In [ ]:
# Create RAG pipeline
# Use document IDs from qrels (relevant documents in the index)
doc_id_list = list(relevant_doc_ids)

rag = RAGPipeline(
    retriever=splade_encoder,
    generator=gen_model,
    generator_tokenizer=gen_tokenizer,
    pt_index=index,
    doc_ids=doc_id_list,
    top_k=3,
)

In [ ]:
# Test the pipeline
test_query = "How do I debug a Python program?"

result = rag(test_query)

print(f"Question: {result['query']}")
print(f"\n{'=' * 80}")
print("Retrieved documents:")
retrieved_texts = get_text_from_index(
    index, [doc_id for doc_id, _ in result["retrieved_docs"]]
)
for doc_id, score in result["retrieved_docs"]:
    print(f"  [{score:.2f}] {retrieved_texts.get(doc_id, '')[:100]}...")
print(f"\n{'=' * 80}")
print(f"Generated answer:\n{result['generated_answer']}")

## Part 6: Complete Evaluation

Let's compare BM25 and SPLADE on our test dataset.

### Complete Evaluation

In [ ]:
def retriever_to_results(
    retriever_fn,
    queries_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert a retriever function to a results DataFrame for ir-measures.

    Args:
        retriever_fn: Function that takes a query and returns [(doc_id, score)]
        queries_df: DataFrame with queries

    Returns:
        DataFrame with columns [qid, docno, score]
    """
    all_results = []

    for _, query_row in tqdm(
        queries_df.iterrows(), desc="Retrieving", total=len(queries_df)
    ):
        qid = query_row["qid"]
        query_text = query_row["query"]

        results = retriever_fn(query_text)
        for doc_id, score in results:
            all_results.append({"qid": qid, "docno": doc_id, "score": score})

    return pd.DataFrame(all_results)

In [ ]:
# Evaluate SPLADE
test_queries = queries_df.tail(20)  # Use last queries as test



def splade_retriever(query: str) -> List[Tuple[str, float]]:
    return rag.retrieve(query, top_k=20)


splade_results = retriever_to_results(splade_retriever, test_queries)
splade_metrics = ir_measures.calc_aggregate(
    metrics, qrels_ir, to_ir_measures_run(splade_results)
)

print("\n=== SPLADE Results ===")
for metric, value in splade_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Compare with BM25
bm25_results = bm25.transform(test_queries[["qid", "query"]])
bm25_metrics = ir_measures.calc_aggregate(
    metrics, qrels_ir, to_ir_measures_run(bm25_results)
)

print("\n=== BM25 Results ===")
for metric, value in bm25_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Comparison table
print("\n=== Comparison ===")
print(f"{'Metric':<12} {'BM25':>10} {'SPLADE':>10} {'Diff':>10}")
print("-" * 44)
for metric in splade_metrics:
    bm25_val = bm25_metrics[metric]
    splade_val = splade_metrics[metric]
    diff = splade_val - bm25_val
    diff_str = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    print(f"{metric!s:<12} {bm25_val:>10.4f} {splade_val:>10.4f} {diff_str:>10}")